https://medium.com/@sauravjoshi23/complex-query-resolution-through-llamaindex-utilizing-recursive-retrieval-document-agents-and-sub-d4861ecd54e6
https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents-v1.html#build-retriever-enabled-openai-agent

# Dataset Loader

In [ ]:
from islab.nlp.cancer_registry.datasets.utils import CRCPlainTextReader

reader = CRCPlainTextReader()

sample_path = 'crc'
patients = reader.load_data(sample_path)
patients

In [ ]:
len(patients)

# Document Agent and Vector Index Engine based on Auto-merging Retrieval

In [ ]:
from llama_index.prompts.base import PromptTemplate
from llama_index.llms import HuggingFaceLLM
import torch
from transformers import BitsAndBytesConfig

#selected_model = "meta-llama/Llama-2-7b-chat-hf"
selected_model = "mistralai/Mistral-7B-Instruct-v0.1"

system_prompt = '''As a cancer registrar in this system, your primary responsibility is to accurately curate the cancer registry data in compliance with user-defined requirements. It's imperative that your data compilation strictly adheres to the options and categories specified by users. In situations where the correct code cannot be determined due to insufficient information, or the content is unrelated to the specific cancer type in question, your response should be 'NA'. All responses must be formatted exclusively as valid JSON objects. Outputs in any format other than JSON are not acceptable. Each response should include the key 'explan', containing a string that justifies your classification choice. Additionally, incorporate other keys as required by user specifications. Remember, the structure and content of your JSON response should reflect the precision and adherence to guidelines essential for this role.'''

query_wrapper_prompt = PromptTemplate(
    "[INST]<<SYS>>\n" + system_prompt + "<</SYS>>\n\n{query_str}[/INST]"
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

doc_idx_llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=2048,    
    generate_kwargs={"temperature": 0.001, "do_sample": True},
    #query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name=selected_model,
    model_name=selected_model,
    device_map="auto", 
    system_prompt = system_prompt,   
    query_wrapper_prompt=query_wrapper_prompt,
    model_kwargs={"quantization_config": quantization_config, "pad_token_id" : 2},
)

# Construct Index

## Agent Service Context

In [ ]:
from llama_index.llms.llama_utils import messages_to_prompt, completion_to_prompt
from llama_index.llms import HuggingFaceLLM
from llama_index import ServiceContext

agent_system_prompt = '''As a cancer registrar agent, your primary responsibility is to accurately curate the cancer registry data in compliance with user-defined requirements and the document you manage. It's imperative that your data compilation strictly adheres to the options and categories specified by users. In situations where the correct code cannot be determined due to insufficient information, or the content is unrelated to the specific cancer type in question, your response should be 'NA'. All responses must be formatted exclusively as valid JSON objects. Outputs in any format other than JSON are not acceptable. Each response should include the key 'explan', containing a string that justifies your classification choice. Additionally, incorporate other keys as required by user specifications. Remember, the structure and content of your JSON response should reflect the precision and adherence to guidelines essential for this role.'''

agent_llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=2048,    
    generate_kwargs={"temperature": 0.001, "do_sample": True},
    tokenizer_name=selected_model,
    model_name=selected_model,
    is_chat_model = True,

    system_prompt=agent_system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    
    
    # Here is how to fix the alternative error show above.
    # We transform inputs into Llama2 format here
    messages_to_prompt = messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto", 
    model_kwargs={"quantization_config": quantization_config, "pad_token_id" : 2},
)

agent_service_context = ServiceContext.from_defaults(
    llm=agent_llm,
    embed_model="local:BAAI/bge-large-en-v1.5"
)

In [ ]:
# llama_template_laterality ='''Your task is to determine the laterality of lung cancer based on the primary tumor site. The laterality refers to whether the cancer originates from a paired organ and is applicable only to primary tumors. When responding, please use the following numerical options:

# - 0: The cancer does not originate from a paired organ.
# - 1: The primary origin of the cancer is on the right side.
# - 2: The primary origin of the cancer is on the left side.

# Note: Ensure that your response is limited to one of these three options. For instance, if the primary tumor site is in the right lung, your response must be "1". After providing the code, please add two newline characters and then explain the reason for your classification choice.'''

# llama_template_primary_site ='''Your task is to identify and confirm the primary site of lung cancer. The primary site usually refers to specific regions within the respiratory system. When providing your response, please choose from the following options, which correspond to the standard coding for different lung cancer sites:

# - C339: Trachea
# - C340: Main bronchus
# - C341: Upper lobe, lung
# - C342: Middle lobe, lung
# - C343: Lower lobe, lung
# - C348: Overlapping lesion of lung
# - C349: Lung NOS (Not Otherwise Specified)

# Your response should be limited to one of these codes, based on the most accurate location of the primary cancer site. For example, if the primary site is in the upper lobe of the lung, your response must be 'C341'. After providing the code, please add two newline characters and then explain the reason for your classification choice.'''

ccr_histology_template ='''Your primary task is to accurately curate the histology code for patients diagnosed with colorectal cancer. This involves meticulously reviewing and analyzing the given medical record of the patient to determine the most appropriate histology code. Your decision should be based on a thorough assessment of clinical findings and any relevant medical descriptions that shed light on the histological nature of the colorectal cancer. It is essential to use only the information provided in the document at hand.
When responding, you are required to select the most appropriate option from the following standard codes, which represents different histology types of colorectal cancer:

- 8000: Use this code if the histology type is "Neoplasm, malignant", "Blastoma, NOS", "Tumor, malignant, NOS", "Cancer", "Malignancy", "Unclassified tumor, malignant".
- 8010: Use this code if the histology type is "Carcinoma in situ, NOS", "Intraepithelial carcinoma, NOS", "Carcinoma, NOS", "Epithelial tumor, malignant", or "Heterotopia-associated carcinoma".
- 8013: Use this code if the histology type is "Large cell neuroendocrine carcinoma", or "Combined large cell neuroendocrine carcinoma".
- 8014: Use this code if the histology type is "Large cell carcinoma with rhabdoid phenotype".
- 8020: Use this code if the histology type is "Carcinoma, undifferentiated, NOS", "Carcinoma, poorly differentiated, NOS", "Anaplastic undifferentiated carcinoma", or "Dedifferentiated carcinoma".
- 8070: Use this code if the histology type is "squamous cell carcinoma in situ, NOS", "epidermoid carcinoma in situ, NOS", "intraepidermal carcinoma, NOS", "intraepithelial squamous cell carcinoma", "squamous cell carcinoma, NOS", "squamous cell carcinoma, usual type", "epidermoid carcinoma, NOS", "squamous carcinoma", or "squamous cell epithelioma".
- 8120: Use this code if the histology type is "Transitional cell carcinoma in situ", "Urothelial carcinoma in situ", "Transitional cell carcinoma, NOS", "Urothelial carcinoma, NOS", or "Transitional carcinoma".
- 8140: Use this code if the histology type is "Adenocarcinoma in situ, NOS", "Adenocarcinoma, NOS", "Adenocarcinoma, usual type", "Carcinoma of Skene, Cowper and Littre Glands", "Endolymphatic sac tumor", "Parathyroid carcinoma", or "Acinar adenocarcinoma of prostate".
- 8210: Use this code if the histology type is "Adenocarcinoma in situ in adenomatous polyp", "Adenocarcinoma in situ in a polyp, NOS", "Carcinoma in situ in a polyp, NOS", "Adenocarcinoma in situ in polypoid adenoma", "Adenocarcinoma in situ in tubular adenoma", "Carcinoma in situ in adenomatous polyp", "Adenocarcinoma in adenomatous polyp", "Adenocarcinoma in a polyp, NOS", "Carcinoma in a polyp, NOS", or "Adenocarcinoma in polypoid adenoma".
- 8211: Use this code if the histology type is "Tubular adenocarcinoma", or "Tubular carcinoma".
- 8244: Use this code if the histology type is "Mixed adenoneuroendocrine carcinoma", "Combined carcinoid and adenocarcinoma", "Combined/mixed carcinoid and adenocarcinoma", "Composite carcinoid", "MANEC", or "Mixed carcinoid-adenocarcinoma"
- 8246: Use this code if the histology type is "Neuroendocrine carcinoma, NOS", or "Poorly differentiated neuroendocrine neoplasm".
- 8261: Use this code if the histology type is "Adenocarcinoma in situ in villous adenoma", or "Adenocarcinoma in villous adenoma".
- 8263: Use this code if the histology type is "Adenocarcinoma in situ in tubulovillous adenoma", "Adenocarcinoma in tubulovillous adenoma", "Villoglandular carcinoma", "Villoglandular variant of endometrioid adenocarcinoma", "Papillotubular adenocarcinoma", or "Tubulopapillary adenocarcinoma".
- 8480: Use this code if the histology type is "Mucinous adenocarcinoma", "Acinar adenocarcinoma, mucinous variant", "Colloid adenocarcinoma", "Colloid carcinoma", "Gelatinous adenocarcinoma", "Gelatinous carcinoma", "Mucinous carcinoma", "Mucoid adenocarcinoma", "Mucoid carcinoma", "Mucous adenocarcinoma", "Mucous carcinoma", "Pseudomyxoma peritonei with unknown primary site", "Mucinous tubular and spindle cell carcinoma".
- 8490: Use this code if the histology type is "Signet ring cell carcinoma", "Signet ring cell adenocarcinoma", "Acinar adenocarcinoma, signet ring-like variant", "Mucinous carcinoma, signet ring cell type", "Poorly cohesive carcinoma", or "Signet ring cell/histiocytoid carcinoma".
- 8510: Use this code if the histology type is "Medullary carcinoma, NOS", "Medullary adenocarcinoma", or "Medullary-like carcinoma".

Your response should be limited to one of these codes, aligning with the most accurate and specific histological classification of the cancer case. For example, if the case is pathologically diagnosed as 'Adenocarcinoma in situ in adenomatous polyp', your response should be in the following JSON format:

{
"explan": "[Your explanation here based on the document]",
"cite": "[Cited passages from the document used for your decision]",
"histology": "8210"
}

Alternatively, if the case is diagnosed as 'Adenocarcinoma in situ' without further specification, your response should be:

{
"explan": "[Your explanation here based on the document]",
"cite": "[Cited passages from the document used for your decision]",
"histology": "8140"
}'''

# llama_template_behavior = '''Your task is to record the behavior code in the pathology diagnosis. The behavior code is represented by the 5th code in the M-code system. When providing your response, please choose from the following options, which correspond to the standard behavior coding used by pathologist:

# - N/A: Use this code if there is no information provided in the pathology report.
# - 0: Benign
# - 1: Borderline
# - 2: In situ
# - 3: Malignant, primary site
# - 6: Malignant, metastatic site
# - 9: Malignant, uncertain whether primary or metastatic site

# Please ensure your response is limited to one of these specific codes. For example, if a tumor's code is M-8070/3, your response must be '3' indicaing that the tumor is malignant and located at the primary site, as denoted by the /3 behavior code. After providing the code, please add two newline characters and then explain the reason for your classification choice.'''

# llama_template_node_examine = '''Your task is to record the total number of regional lymph nodes examined by the pathologist in each case. This data is crucial for understanding the extent of lymph node examination in cancer cases. When coding, focus on the total count of lymph nodes examined, regardless of whether they are positive or negative for malignancy.

# Here are two examples to guide you:

# Example 1: The report states 'Lymph node, regional, lymphadenectomy, negative for malignancy (0/20)'. The code for this case is '20', representing the total number of lymph nodes examined.
# Example 2: The report says 'Lymph node, apical, lymphadenectomy, negative for malignancy (0/9)'. In this case, the correct code is '9', indicating the total count of lymph nodes examined.
# After providing the code, add two newline characters and then provide a detailed explanation for your coding decision. This explanation should include the rationale behind selecting the specific count based on the information provided in the pathology report.'''

# llama_template_node_positive='''Your task is to record the total number of regional lymph nodes that tested positive for malignancy as examined by the pathologist. This information is crucial for assessing the extent of cancer involvement in lymph nodes.
# When providing your answer, focus on the count of lymph nodes that were positive for malignancy. Here are two examples to illustrate the process:
# Example 1: The pathology report states 'Lymph node, regional, lymphadenectomy, negative for malignancy (12/20)'. The answer in this case should be '12', as there were 12 lymph nodes testing positive for malignancy out of the 20 examined.
# Example 2: The report reads 'Lymph node, apical, lymphadenectomy, negative for malignancy (0/9)'. Here, the correct answer is also '0', indicating that none of the 9 examined lymph nodes were positive for malignancy.
# After providing the answer, add two newline characters and then provide a detailed explanation of your reasoning. This explanation should clearly articulate why you selected the specific count based on the pathology report's findings.'''

# llama_template_path_TNM = '''Your task is to response the cancer staging information in terms of PATH_T, PATH_N, and PATH_M based on the pathological TNM criteria. The TNM staging system is crucial for assessing trends in cancer treatment and control. It is used by physicians for prognosis estimation, treatment planning, evaluation of therapies, outcome analysis, follow-up planning, and assessing early detection results. Pathological staging is primarily determined by pathologists, with input from other medical specialists. It is usually written in the format of T(PATH_T code)N(PATH_N code)[M(PATH_M code)], where the PATH_M code may be omitted. For example, in the description 'T1b1N2cM0', the PATH_T, PATH_N, and PATH_M codes are 1b, 2c, and 0, respectively. In cases where no explicit description is provided by physicians, you should make informed judgments based on available information in the medical records.
# Here is the detail definition for the three codes.
# - PATH_T: This code indicates the size or extent of the primary tumor.
# -- Available codes: X, 0, A, IS, ISU, ISD, ISDC, ISPA, ISLA, 1M, 2, 2A, 2A1, 2A2, 2B, 2C, 2D, 3, 3A, 3B, 1, 1A, 1A1, 1A2, 1B, 1B1, 1B2, 1B3, 1C, 1C1, 1C2, 1C3, 1D, 3C, 3D, 4, 4A, 4B, 4C, 4D, 4E, 8888 (Not applicable), 9999 (Not recorded)
# - PATH_N: This code represents the presence and extent of regional lymph node metastasis.
# -- Available codes: X, 0, 0A, 0B, 0C, 0D, 1, 1A, 1AS, 1B, 1C, 1M, 2M, 2, 2A, 2B, 2C, 3, 3A, 3B, 3C, 888 (Not applicable), 999 (Not recorded)
# - PATH_M: This code indicates the presence of distant metastasis.
# -- Available codes: 1, 1A, 1A0, 1A1, 1B, 1B0, 1B1, 1C, 1C0, 1C1, 1D, 1D0, C, CA, CA0, CA1, CB, CB0, CB1, CC, CC0, CC1, CD, CD0, CD1, CE, 888 (Not applicable), 999 (Not recorded)

# Please ensure your response is limited to one of these specific codes. For example, if the report states 'pT3N0M2b', your coding results in this case should be:
# PATH_T: 3
# PATH_N: 0
# PATH_M: 2B
# After providing the code, please add two newline characters and then explain the reason for your classification choice.'''
# # Here are some examples to guide you:
# # Example 1: The pathology report states 'T1a1N1'. Your coding results in this case should be:
# # PATH_T: 1A1
# # PATH_N: 1

# # Example 2: The pathology report states 'pT3N0M2b'. Your coding results in this case should be:
# # Results:
# # PATH_T: 3
# # PATH_N: 0
# # PATH_M: 2B

# # After providing the codes, add two newline characters and then elaborate on the reasoning behind your coding results. This explanation should clearly articulate how the information from the medical records or pathology reports influenced your coding decisions.'''

# llama_templates = [#{'role': 'user', 'content': llama_template_laterality}, {'role': 'user', 'content': llama_template_primary_site},
#                   #  {'role': 'user', 'content': llama_template_path_TNM},
#                 #    {'role': 'user', 'content': llama_template_behavior}, {'role': 'user', 'content': llama_template_node_examine},
#                 #    {'role': 'user', 'content': llama_template_node_positive},
#                     {'role': 'user', 'content': llama_template_histology},
#                     {'role': 'user', 'content': llama_template_primary_site_laterality}]

agent_document_idx_templates = {
    "histology": ccr_histology_template,
    #"primary_site_and_laterality": llama_template_primary_site_laterality
}

def update_query_engine_prompts(tool_prompt, query_engine):
    qa_prompt_tmpl_str = """Context information is below.
---------------------
{context_str}
---------------------
Given the context information and the following knowledge and command, answer the query.
Knowledge and command: {cancer_registry_kg}
Query: {query_str}
Answer: \
"""
    prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)
    partial_prompt_tmpl = prompt_tmpl.partial_format(cancer_registry_kg=tool_prompt)
    print(partial_prompt_tmpl)
    query_engine.update_prompts({"response_synthesizer:text_qa_template": partial_prompt_tmpl})

    refine_prompt_tmpl_str = """The original query is as follows: {query_str}
We have provided an existing answer: {existing_answer}
Knowledge and command: {cancer_registry_kg}
We have the opportunity to refine the existing answer (only if needed) with some more context below.
------------
{context_msg}
------------
Given the new context, refine the original answer to better answer the query. If the context isn't useful, return the original answer.
Refined Answer:\
"""
    prompt_tmpl = PromptTemplate(refine_prompt_tmpl_str)
    partial_prompt_tmpl = prompt_tmpl.partial_format(cancer_registry_kg=tool_prompt)
    print(partial_prompt_tmpl)
    query_engine.update_prompts({"response_synthesizer:refine_template": partial_prompt_tmpl})


## Query Engine

See https://platform.openai.com/docs/guides/prompt-engineering?utm_source=ai.briefnewsletter.com&utm_medium=newsletter&utm_campaign=2023-ai

In [ ]:
"""ReAct agent worker."""

import asyncio
import uuid
from itertools import chain
from threading import Thread
from typing import (
    Any,
    AsyncGenerator,
    Dict,
    Generator,
    List,
    Optional,
    Sequence,
    Tuple,
    cast,
)

from llama_index.agent.react.formatter import ReActChatFormatter
from llama_index.agent.react.output_parser import ReActOutputParser
from llama_index.agent.react.types import (
    ActionReasoningStep,
    BaseReasoningStep,
    ObservationReasoningStep,
    ResponseReasoningStep,
)
from llama_index.agent.types import (
    BaseAgentWorker,
    Task,
    TaskStep,
    TaskStepOutput,
)
from llama_index.callbacks import (
    CallbackManager,
    CBEventType,
    EventPayload,
    trace_method,
)
from llama_index.chat_engine.types import (
    AGENT_CHAT_RESPONSE_TYPE,
    AgentChatResponse,
    StreamingAgentChatResponse,
)
from llama_index.llms.base import ChatMessage, ChatResponse
from llama_index.llms.llm import LLM
from llama_index.llms.openai import OpenAI
from llama_index.llms.types import MessageRole
from llama_index.memory.chat_memory_buffer import ChatMemoryBuffer
from llama_index.memory.types import BaseMemory
from llama_index.objects.base import ObjectRetriever
from llama_index.tools import BaseTool, ToolOutput, adapt_to_async_tool
from llama_index.tools.types import AsyncBaseTool
from llama_index.utils import print_text, unit_generator

DEFAULT_MODEL_NAME = "gpt-3.5-turbo-0613"


def add_user_step_to_reasoning(
    step: TaskStep,
    memory: BaseMemory,
    current_reasoning: List[BaseReasoningStep],
    verbose: bool = False,
) -> None:
    """Add user step to memory."""
    if "is_first" in step.step_state and step.step_state["is_first"]:
        # add to new memory
        memory.put(ChatMessage(content=step.input, role=MessageRole.USER))
        step.step_state["is_first"] = False
    else:
        reasoning_step = ObservationReasoningStep(observation=step.input)
        current_reasoning.append(reasoning_step)
        if verbose:
            print(f"Added user message to memory: {step.input}")


class ReActAgentWorker(BaseAgentWorker):
    """OpenAI Agent worker."""

    def __init__(
        self,
        tools: Sequence[BaseTool],
        llm: LLM,
        max_iterations: int = 10,
        react_chat_formatter: Optional[ReActChatFormatter] = None,
        output_parser: Optional[ReActOutputParser] = None,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = False,
        tool_retriever: Optional[ObjectRetriever[BaseTool]] = None,
    ) -> None:
        self._llm = llm
        self.callback_manager = callback_manager or llm.callback_manager
        self._max_iterations = max_iterations
        self._react_chat_formatter = react_chat_formatter or ReActChatFormatter()
        self._output_parser = output_parser or ReActOutputParser()
        self._verbose = verbose

        if len(tools) > 0 and tool_retriever is not None:
            raise ValueError("Cannot specify both tools and tool_retriever")
        elif len(tools) > 0:
            self._get_tools = lambda _: tools
        elif tool_retriever is not None:
            tool_retriever_c = cast(ObjectRetriever[BaseTool], tool_retriever)
            self._get_tools = lambda message: tool_retriever_c.retrieve(message)
        else:
            self._get_tools = lambda _: []

    @classmethod
    def from_tools(
        cls,
        tools: Optional[Sequence[BaseTool]] = None,
        tool_retriever: Optional[ObjectRetriever[BaseTool]] = None,
        llm: Optional[LLM] = None,
        max_iterations: int = 10,
        react_chat_formatter: Optional[ReActChatFormatter] = None,
        output_parser: Optional[ReActOutputParser] = None,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = False,
        **kwargs: Any,
    ) -> "ReActAgentWorker":
        """Convenience constructor method from set of of BaseTools (Optional).

        NOTE: kwargs should have been exhausted by this point. In other words
        the various upstream components such as BaseSynthesizer (response synthesizer)
        or BaseRetriever should have picked up off their respective kwargs in their
        constructions.

        Returns:
            ReActAgent
        """
        llm = llm or OpenAI(model=DEFAULT_MODEL_NAME)
        if callback_manager is not None:
            llm.callback_manager = callback_manager
        return cls(
            tools=tools or [],
            tool_retriever=tool_retriever,
            llm=llm,
            max_iterations=max_iterations,
            react_chat_formatter=react_chat_formatter,
            output_parser=output_parser,
            callback_manager=callback_manager,
            verbose=verbose,
        )

    def initialize_step(self, task: Task, **kwargs: Any) -> TaskStep:
        """Initialize step from task."""
        sources: List[ToolOutput] = []
        current_reasoning: List[BaseReasoningStep] = []
        # temporary memory for new messages
        new_memory = ChatMemoryBuffer.from_defaults()

        # initialize task state
        task_state = {
            "sources": sources,
            "current_reasoning": current_reasoning,
            "new_memory": new_memory,
        }
        task.extra_state.update(task_state)

        return TaskStep(
            task_id=task.task_id,
            step_id=str(uuid.uuid4()),
            input=task.input,
            step_state={"is_first": True},
        )

    def get_tools(self, input: str) -> List[AsyncBaseTool]:
        """Get tools."""
        return [adapt_to_async_tool(t) for t in self._get_tools(input)]

    def _extract_reasoning_step(
        self, output: ChatResponse, is_streaming: bool = False
    ) -> Tuple[str, List[BaseReasoningStep], bool]:
        """
        Extracts the reasoning step from the given output.

        This method parses the message content from the output,
        extracts the reasoning step, and determines whether the processing is
        complete. It also performs validation checks on the output and
        handles possible errors.
        """
        if output.message.content is None:
            raise ValueError("Got empty message.")
        message_content = output.message.content
        current_reasoning = []
        try:
            reasoning_step = self._output_parser.parse(message_content, is_streaming)
        except BaseException as exc:
            raise ValueError(f"Could not parse output: {message_content}") from exc
        if self._verbose:
            print_text(f"{reasoning_step.get_content()}\n", color="pink")
        current_reasoning.append(reasoning_step)

        if reasoning_step.is_done:
            return message_content, current_reasoning, True

        reasoning_step = cast(ActionReasoningStep, reasoning_step)
        if not isinstance(reasoning_step, ActionReasoningStep):
            raise ValueError(f"Expected ActionReasoningStep, got {reasoning_step}")

        return message_content, current_reasoning, False

    def _process_actions(
        self,
        task: Task,
        tools: Sequence[AsyncBaseTool],
        output: ChatResponse,
        is_streaming: bool = False,
    ) -> Tuple[List[BaseReasoningStep], bool]:
        tools_dict: Dict[str, AsyncBaseTool] = {
            tool.metadata.get_name(): tool for tool in tools
        }
        _, current_reasoning, is_done = self._extract_reasoning_step(
            output, is_streaming
        )

        if is_done:
            return current_reasoning, True

        # call tool with input
        reasoning_step = cast(ActionReasoningStep, current_reasoning[-1])
        tool = tools_dict[reasoning_step.action]
        with self.callback_manager.event(
            CBEventType.FUNCTION_CALL,
            payload={
                EventPayload.FUNCTION_CALL: reasoning_step.action_input,
                EventPayload.TOOL: tool.metadata,
            },
        ) as event:
            tool_output = tool.call(**reasoning_step.action_input)
            event.on_end(payload={EventPayload.FUNCTION_OUTPUT: str(tool_output)})

        task.extra_state["sources"].append(tool_output)

        observation_step = ObservationReasoningStep(observation=str(tool_output))
        current_reasoning.append(observation_step)
        if self._verbose:
            print_text(f"{observation_step.get_content()}\n", color="blue")
        return current_reasoning, False

    async def _aprocess_actions(
        self,
        task: Task,
        tools: Sequence[AsyncBaseTool],
        output: ChatResponse,
        is_streaming: bool = False,
    ) -> Tuple[List[BaseReasoningStep], bool]:
        tools_dict = {tool.metadata.name: tool for tool in tools}
        _, current_reasoning, is_done = self._extract_reasoning_step(
            output, is_streaming
        )

        if is_done:
            return current_reasoning, True

        # call tool with input
        reasoning_step = cast(ActionReasoningStep, current_reasoning[-1])
        tool = tools_dict[reasoning_step.action]
        with self.callback_manager.event(
            CBEventType.FUNCTION_CALL,
            payload={
                EventPayload.FUNCTION_CALL: reasoning_step.action_input,
                EventPayload.TOOL: tool.metadata,
            },
        ) as event:
            tool_output = await tool.acall(**reasoning_step.action_input)
            event.on_end(payload={EventPayload.FUNCTION_OUTPUT: str(tool_output)})

        task.extra_state["sources"].append(tool_output)

        observation_step = ObservationReasoningStep(observation=str(tool_output))
        current_reasoning.append(observation_step)
        if self._verbose:
            print_text(f"{observation_step.get_content()}\n", color="blue")
        return current_reasoning, False

    def _get_response(
        self,
        current_reasoning: List[BaseReasoningStep],
        sources: List[ToolOutput],
    ) -> AgentChatResponse:
        """Get response from reasoning steps."""
        if len(current_reasoning) == 0:
            raise ValueError("No reasoning steps were taken.")
        elif len(current_reasoning) == self._max_iterations:
            raise ValueError("Reached max iterations.")

        if isinstance(current_reasoning[-1], ResponseReasoningStep):
            response_step = cast(ResponseReasoningStep, current_reasoning[-1])
            response_str = response_step.response
        else:
            response_str = current_reasoning[-1].get_content()

        # TODO: add sources from reasoning steps
        return AgentChatResponse(response=response_str, sources=sources)

    def _get_task_step_response(
        self, agent_response: AGENT_CHAT_RESPONSE_TYPE, step: TaskStep, is_done: bool
    ) -> TaskStepOutput:
        """Get task step response."""
        if is_done:
            new_steps = []
        else:
            new_steps = [
                step.get_next_step(
                    step_id=str(uuid.uuid4()),
                    # NOTE: input is unused
                    input=None,
                )
            ]

        return TaskStepOutput(
            output=agent_response,
            task_step=step,
            is_last=is_done,
            next_steps=new_steps,
        )

    def _infer_stream_chunk_is_final(self, chunk: ChatResponse) -> bool:
        """Infers if a chunk from a live stream is the start of the final
        reasoning step. (i.e., and should eventually become
        ResponseReasoningStep — not part of this function's logic tho.).

        Args:
            chunk (ChatResponse): the current chunk stream to check

        Returns:
            bool: Boolean on whether the chunk is the start of the final response
        """
        latest_content = chunk.message.content
        if latest_content:
            if not latest_content.startswith(
                "Thought"
            ):  # doesn't follow thought-action format
                return True
            else:
                if "Answer: " in latest_content:
                    return True
        return False

    def _add_back_chunk_to_stream(
        self, chunk: ChatResponse, chat_stream: Generator[ChatResponse, None, None]
    ) -> Generator[ChatResponse, None, None]:
        """Helper method for adding back initial chunk stream of final response
        back to the rest of the chat_stream.

        Args:
            chunk (ChatResponse): the chunk to add back to the beginning of the
                                    chat_stream.

        Return:
            Generator[ChatResponse, None, None]: the updated chat_stream
        """
        updated_stream = chain.from_iterable(  # need to add back partial response chunk
            [
                unit_generator(chunk),
                chat_stream,
            ]
        )
        # use cast to avoid mypy issue with chain and Generator
        updated_stream_c: Generator[ChatResponse, None, None] = cast(
            Generator[ChatResponse, None, None], updated_stream
        )
        return updated_stream_c

    async def _async_add_back_chunk_to_stream(
        self, chunk: ChatResponse, chat_stream: AsyncGenerator[ChatResponse, None]
    ) -> AsyncGenerator[ChatResponse, None]:
        """Helper method for adding back initial chunk stream of final response
        back to the rest of the chat_stream.

        NOTE: this itself is not an async function.

        Args:
            chunk (ChatResponse): the chunk to add back to the beginning of the
                                    chat_stream.

        Return:
            AsyncGenerator[ChatResponse, None]: the updated async chat_stream
        """
        yield chunk
        async for item in chat_stream:
            yield item

    def _run_step(
        self,
        step: TaskStep,
        task: Task,
    ) -> TaskStepOutput:
        """Run step."""
        if step.input is not None:
            add_user_step_to_reasoning(
                step,
                task.extra_state["new_memory"],
                task.extra_state["current_reasoning"],
                verbose=self._verbose,
            )
        # TODO: see if we want to do step-based inputs
        tools = self.get_tools(task.input)

        input_chat = self._react_chat_formatter.format(
            tools,
            chat_history=task.memory.get() + task.extra_state["new_memory"].get_all(),
            current_reasoning=task.extra_state["current_reasoning"],
        )

        # send prompt
        chat_response = self._llm.chat(input_chat)
        # given react prompt outputs, call tools or return response
        reasoning_steps, is_done = self._process_actions(
            task, tools, output=chat_response
        )
        task.extra_state["current_reasoning"].extend(reasoning_steps)
        agent_response = self._get_response(
            task.extra_state["current_reasoning"], task.extra_state["sources"]
        )
        if is_done:
            task.extra_state["new_memory"].put(
                ChatMessage(content=agent_response.response, role=MessageRole.ASSISTANT)
            )

        return self._get_task_step_response(agent_response, step, is_done)

    async def _arun_step(
        self,
        step: TaskStep,
        task: Task,
    ) -> TaskStepOutput:
        """Run step."""
        if step.input is not None:
            add_user_step_to_reasoning(
                step,
                task.extra_state["new_memory"],
                task.extra_state["current_reasoning"],
                verbose=self._verbose,
            )
        # TODO: see if we want to do step-based inputs
        tools = self.get_tools(task.input)

        input_chat = self._react_chat_formatter.format(
            tools,
            chat_history=task.memory.get() + task.extra_state["new_memory"].get_all(),
            current_reasoning=task.extra_state["current_reasoning"],
        )
        # send prompt
        chat_response = await self._llm.achat(input_chat)
        # given react prompt outputs, call tools or return response
        reasoning_steps, is_done = await self._aprocess_actions(
            task, tools, output=chat_response
        )
        task.extra_state["current_reasoning"].extend(reasoning_steps)
        agent_response = self._get_response(
            task.extra_state["current_reasoning"], task.extra_state["sources"]
        )
        if is_done:
            task.extra_state["new_memory"].put(
                ChatMessage(content=agent_response.response, role=MessageRole.ASSISTANT)
            )

        return self._get_task_step_response(agent_response, step, is_done)

    def _run_step_stream(
        self,
        step: TaskStep,
        task: Task,
    ) -> TaskStepOutput:
        """Run step."""
        if step.input is not None:
            add_user_step_to_reasoning(
                step,
                task.extra_state["new_memory"],
                task.extra_state["current_reasoning"],
                verbose=self._verbose,
            )
        # TODO: see if we want to do step-based inputs
        tools = self.get_tools(task.input)

        input_chat = self._react_chat_formatter.format(
            tools,
            chat_history=task.memory.get() + task.extra_state["new_memory"].get_all(),
            current_reasoning=task.extra_state["current_reasoning"],
        )

        chat_stream = self._llm.stream_chat(input_chat)

        # iterate over stream, break out if is final answer after the "Answer: "
        full_response = ChatResponse(
            message=ChatMessage(content=None, role="assistant")
        )
        is_done = False
        for latest_chunk in chat_stream:
            full_response = latest_chunk
            is_done = self._infer_stream_chunk_is_final(latest_chunk)
            if is_done:
                break

        if not is_done:
            # given react prompt outputs, call tools or return response
            reasoning_steps, _ = self._process_actions(
                task, tools=tools, output=full_response, is_streaming=True
            )
            task.extra_state["current_reasoning"].extend(reasoning_steps)
            # use _get_response to return intermediate response
            agent_response: AGENT_CHAT_RESPONSE_TYPE = self._get_response(
                task.extra_state["current_reasoning"], task.extra_state["sources"]
            )
        else:
            # Get the response in a separate thread so we can yield the response
            response_stream = self._add_back_chunk_to_stream(
                chunk=latest_chunk, chat_stream=chat_stream
            )

            agent_response = StreamingAgentChatResponse(
                chat_stream=response_stream,
                sources=task.extra_state["sources"],
            )
            thread = Thread(
                target=agent_response.write_response_to_history,
                args=(task.extra_state["new_memory"],),
            )
            thread.start()

        return self._get_task_step_response(agent_response, step, is_done)

    async def _arun_step_stream(
        self,
        step: TaskStep,
        task: Task,
    ) -> TaskStepOutput:
        """Run step."""
        if step.input is not None:
            add_user_step_to_reasoning(
                step,
                task.extra_state["new_memory"],
                task.extra_state["current_reasoning"],
                verbose=self._verbose,
            )
        # TODO: see if we want to do step-based inputs
        tools = self.get_tools(task.input)

        input_chat = self._react_chat_formatter.format(
            tools,
            chat_history=task.memory.get() + task.extra_state["new_memory"].get_all(),
            current_reasoning=task.extra_state["current_reasoning"],
        )

        chat_stream = await self._llm.astream_chat(input_chat)

        # iterate over stream, break out if is final answer after the "Answer: "
        full_response = ChatResponse(
            message=ChatMessage(content=None, role="assistant")
        )
        is_done = False
        async for latest_chunk in chat_stream:
            full_response = latest_chunk
            is_done = self._infer_stream_chunk_is_final(latest_chunk)
            if is_done:
                break

        if not is_done:
            # given react prompt outputs, call tools or return response
            reasoning_steps, _ = self._process_actions(
                task, tools=tools, output=full_response, is_streaming=True
            )
            task.extra_state["current_reasoning"].extend(reasoning_steps)
            # use _get_response to return intermediate response
            agent_response: AGENT_CHAT_RESPONSE_TYPE = self._get_response(
                task.extra_state["current_reasoning"], task.extra_state["sources"]
            )
        else:
            # Get the response in a separate thread so we can yield the response
            response_stream = self._async_add_back_chunk_to_stream(
                chunk=latest_chunk, chat_stream=chat_stream
            )

            agent_response = StreamingAgentChatResponse(
                achat_stream=response_stream,
                sources=task.extra_state["sources"],
            )
            # create task to write chat response to history
            asyncio.create_task(
                agent_response.awrite_response_to_history(
                    task.extra_state["new_memory"]
                )
            )
            # wait until response writing is done
            await agent_response._is_function_false_event.wait()

        return self._get_task_step_response(agent_response, step, is_done)

    @trace_method("run_step")
    def run_step(self, step: TaskStep, task: Task, **kwargs: Any) -> TaskStepOutput:
        """Run step."""
        return self._run_step(step, task)

    @trace_method("run_step")
    async def arun_step(
        self, step: TaskStep, task: Task, **kwargs: Any
    ) -> TaskStepOutput:
        """Run step (async)."""
        return await self._arun_step(step, task)

    @trace_method("run_step")
    def stream_step(self, step: TaskStep, task: Task, **kwargs: Any) -> TaskStepOutput:
        """Run step (stream)."""
        # TODO: figure out if we need a different type for TaskStepOutput
        return self._run_step_stream(step, task)

    @trace_method("run_step")
    async def astream_step(
        self, step: TaskStep, task: Task, **kwargs: Any
    ) -> TaskStepOutput:
        """Run step (async stream)."""
        return await self._arun_step_stream(step, task)

    def finalize_task(self, task: Task, **kwargs: Any) -> None:
        """Finalize task, after all the steps are completed."""
        # add new messages to memory
        task.memory.set(task.memory.get() + task.extra_state["new_memory"].get_all())
        # reset new memory
        task.extra_state["new_memory"].reset()
        
import uuid
from abc import abstractmethod
from collections import deque
from typing import Any, Deque, Dict, List, Optional, Union, cast

from llama_index.agent.types import (
    BaseAgent,
    BaseAgentWorker,
    Task,
    TaskStep,
    TaskStepOutput,
)
from llama_index.bridge.pydantic import BaseModel, Field
from llama_index.callbacks import (
    CallbackManager,
    CBEventType,
    EventPayload,
    trace_method,
)
from llama_index.chat_engine.types import (
    AGENT_CHAT_RESPONSE_TYPE,
    AgentChatResponse,
    ChatResponseMode,
    StreamingAgentChatResponse,
)
from llama_index.llms.base import ChatMessage
from llama_index.llms.llm import LLM
from llama_index.memory import BaseMemory, ChatMemoryBuffer
from llama_index.memory.types import BaseMemory


class BaseAgentRunner(BaseAgent):
    """Base agent runner."""

    @abstractmethod
    def create_task(self, input: str, **kwargs: Any) -> Task:
        """Create task."""

    @abstractmethod
    def delete_task(
        self,
        task_id: str,
    ) -> None:
        """Delete task.

        NOTE: this will not delete any previous executions from memory.

        """

    @abstractmethod
    def list_tasks(self, **kwargs: Any) -> List[Task]:
        """List tasks."""

    @abstractmethod
    def get_task(self, task_id: str, **kwargs: Any) -> Task:
        """Get task."""

    @abstractmethod
    def get_upcoming_steps(self, task_id: str, **kwargs: Any) -> List[TaskStep]:
        """Get upcoming steps."""

    @abstractmethod
    def get_completed_steps(self, task_id: str, **kwargs: Any) -> List[TaskStepOutput]:
        """Get completed steps."""

    def get_completed_step(
        self, task_id: str, step_id: str, **kwargs: Any
    ) -> TaskStepOutput:
        """Get completed step."""
        # call get_completed_steps, and then find the right task
        completed_steps = self.get_completed_steps(task_id, **kwargs)
        for step_output in completed_steps:
            if step_output.task_step.step_id == step_id:
                return step_output
        raise ValueError(f"Could not find step_id: {step_id}")

    @abstractmethod
    def run_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step."""

    @abstractmethod
    async def arun_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (async)."""

    @abstractmethod
    def stream_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (stream)."""

    @abstractmethod
    async def astream_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (async stream)."""

    @abstractmethod
    def finalize_response(
        self,
        task_id: str,
        step_output: Optional[TaskStepOutput] = None,
    ) -> AGENT_CHAT_RESPONSE_TYPE:
        """Finalize response."""

    @abstractmethod
    def undo_step(self, task_id: str) -> None:
        """Undo previous step."""
        raise NotImplementedError("undo_step not implemented")


def validate_step_from_args(
    task_id: str, input: Optional[str] = None, step: Optional[Any] = None, **kwargs: Any
) -> Optional[TaskStep]:
    """Validate step from args."""
    if step is not None:
        if input is not None:
            raise ValueError("Cannot specify both `step` and `input`")
        if not isinstance(step, TaskStep):
            raise ValueError(f"step must be TaskStep: {step}")
        return step
    else:
        return (
            None
            if input is None
            else TaskStep(
                task_id=task_id, step_id=str(uuid.uuid4()), input=input, **kwargs
            )
        )


class TaskState(BaseModel):
    """Task state."""

    task: Task = Field(..., description="Task.")
    step_queue: Deque[TaskStep] = Field(
        default_factory=deque, description="Task step queue."
    )
    completed_steps: List[TaskStepOutput] = Field(
        default_factory=list, description="Completed step outputs."
    )


class AgentState(BaseModel):
    """Agent state."""

    task_dict: Dict[str, TaskState] = Field(
        default_factory=dict, description="Task dictionary."
    )

    def get_task(self, task_id: str) -> Task:
        """Get task state."""
        return self.task_dict[task_id].task

    def get_completed_steps(self, task_id: str) -> List[TaskStepOutput]:
        """Get completed steps."""
        return self.task_dict[task_id].completed_steps

    def get_step_queue(self, task_id: str) -> Deque[TaskStep]:
        """Get step queue."""
        return self.task_dict[task_id].step_queue


class AgentRunner(BaseAgentRunner):
    """Agent runner.

    Top-level agent orchestrator that can create tasks, run each step in a task,
    or run a task e2e. Stores state and keeps track of tasks.

    Args:
        agent_worker (BaseAgentWorker): step executor
        chat_history (Optional[List[ChatMessage]], optional): chat history. Defaults to None.
        state (Optional[AgentState], optional): agent state. Defaults to None.
        memory (Optional[BaseMemory], optional): memory. Defaults to None.
        llm (Optional[LLM], optional): LLM. Defaults to None.
        callback_manager (Optional[CallbackManager], optional): callback manager. Defaults to None.
        init_task_state_kwargs (Optional[dict], optional): init task state kwargs. Defaults to None.

    """

    # # TODO: implement this in Pydantic

    def __init__(
        self,
        agent_worker: BaseAgentWorker,
        chat_history: Optional[List[ChatMessage]] = None,
        state: Optional[AgentState] = None,
        memory: Optional[BaseMemory] = None,
        llm: Optional[LLM] = None,
        callback_manager: Optional[CallbackManager] = None,
        init_task_state_kwargs: Optional[dict] = None,
        delete_task_on_finish: bool = False,
    ) -> None:
        """Initialize."""
        self.agent_worker = agent_worker
        self.state = state or AgentState()
        self.memory = memory or ChatMemoryBuffer.from_defaults(chat_history, llm=llm)
        self.callback_manager = callback_manager or CallbackManager([])
        self.init_task_state_kwargs = init_task_state_kwargs or {}
        self.delete_task_on_finish = delete_task_on_finish

    @property
    def chat_history(self) -> List[ChatMessage]:
        return self.memory.get_all()

    def reset(self) -> None:
        self.memory.reset()

    def create_task(self, input: str, **kwargs: Any) -> Task:
        """Create task."""
        task = Task(
            input=input,
            memory=self.memory,
            extra_state=self.init_task_state_kwargs,
            **kwargs,
        )
        # # put input into memory
        # self.memory.put(ChatMessage(content=input, role=MessageRole.USER))

        # get initial step from task, and put it in the step queue
        initial_step = self.agent_worker.initialize_step(task)
        task_state = TaskState(
            task=task,
            step_queue=deque([initial_step]),
        )
        # add it to state
        self.state.task_dict[task.task_id] = task_state

        return task

    def delete_task(
        self,
        task_id: str,
    ) -> None:
        """Delete task.

        NOTE: this will not delete any previous executions from memory.

        """
        self.state.task_dict.pop(task_id)

    def list_tasks(self, **kwargs: Any) -> List[Task]:
        """List tasks."""
        return list(self.state.task_dict.values())

    def get_task(self, task_id: str, **kwargs: Any) -> Task:
        """Get task."""
        return self.state.get_task(task_id)

    def get_upcoming_steps(self, task_id: str, **kwargs: Any) -> List[TaskStep]:
        """Get upcoming steps."""
        return list(self.state.get_step_queue(task_id))

    def get_completed_steps(self, task_id: str, **kwargs: Any) -> List[TaskStepOutput]:
        """Get completed steps."""
        return self.state.get_completed_steps(task_id)

    def _run_step(
        self,
        task_id: str,
        step: Optional[TaskStep] = None,
        mode: ChatResponseMode = ChatResponseMode.WAIT,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Execute step."""
        task = self.state.get_task(task_id)
        step_queue = self.state.get_step_queue(task_id)
        step = step or step_queue.popleft()

        # TODO: figure out if you can dynamically swap in different step executors
        # not clear when you would do that by theoretically possible

        if mode == ChatResponseMode.WAIT:
            cur_step_output = self.agent_worker.run_step(step, task, **kwargs)
        elif mode == ChatResponseMode.STREAM:
            cur_step_output = self.agent_worker.stream_step(step, task, **kwargs)
        else:
            raise ValueError(f"Invalid mode: {mode}")
        # append cur_step_output next steps to queue
        next_steps = cur_step_output.next_steps
        step_queue.extend(next_steps)

        # add cur_step_output to completed steps
        completed_steps = self.state.get_completed_steps(task_id)
        completed_steps.append(cur_step_output)

        return cur_step_output

    async def _arun_step(
        self,
        task_id: str,
        step: Optional[TaskStep] = None,
        mode: ChatResponseMode = ChatResponseMode.WAIT,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Execute step."""
        task = self.state.get_task(task_id)
        step_queue = self.state.get_step_queue(task_id)
        step = step or step_queue.popleft()

        # TODO: figure out if you can dynamically swap in different step executors
        # not clear when you would do that by theoretically possible
        if mode == ChatResponseMode.WAIT:
            cur_step_output = await self.agent_worker.arun_step(step, task, **kwargs)
        elif mode == ChatResponseMode.STREAM:
            cur_step_output = await self.agent_worker.astream_step(step, task, **kwargs)
        else:
            raise ValueError(f"Invalid mode: {mode}")
        # append cur_step_output next steps to queue
        next_steps = cur_step_output.next_steps
        step_queue.extend(next_steps)

        # add cur_step_output to completed steps
        completed_steps = self.state.get_completed_steps(task_id)
        completed_steps.append(cur_step_output)

        return cur_step_output

    def run_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step."""
        step = validate_step_from_args(task_id, input, step, **kwargs)
        return self._run_step(task_id, step, mode=ChatResponseMode.WAIT, **kwargs)

    async def arun_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (async)."""
        step = validate_step_from_args(task_id, input, step, **kwargs)
        return await self._arun_step(
            task_id, step, mode=ChatResponseMode.WAIT, **kwargs
        )

    def stream_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (stream)."""
        step = validate_step_from_args(task_id, input, step, **kwargs)
        return self._run_step(task_id, step, mode=ChatResponseMode.STREAM, **kwargs)

    async def astream_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (async stream)."""
        step = validate_step_from_args(task_id, input, step, **kwargs)
        return await self._arun_step(
            task_id, step, mode=ChatResponseMode.STREAM, **kwargs
        )

    def finalize_response(
        self,
        task_id: str,
        step_output: Optional[TaskStepOutput] = None,
    ) -> AGENT_CHAT_RESPONSE_TYPE:
        """Finalize response."""
        if step_output is None:
            step_output = self.state.get_completed_steps(task_id)[-1]
        if not step_output.is_last:
            raise ValueError(
                "finalize_response can only be called on the last step output"
            )

        if not isinstance(
            step_output.output,
            (AgentChatResponse, StreamingAgentChatResponse),
        ):
            raise ValueError(
                "When `is_last` is True, cur_step_output.output must be "
                f"AGENT_CHAT_RESPONSE_TYPE: {step_output.output}"
            )

        # finalize task
        self.agent_worker.finalize_task(self.state.get_task(task_id))

        if self.delete_task_on_finish:
            self.delete_task(task_id)

        return cast(AGENT_CHAT_RESPONSE_TYPE, step_output.output)

    def _chat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
        mode: ChatResponseMode = ChatResponseMode.WAIT,
    ) -> AGENT_CHAT_RESPONSE_TYPE:
        """Chat with step executor."""
        if chat_history is not None:
            self.memory.set(chat_history)
        task = self.create_task(message)

        result_output = None
        while True:
            # pass step queue in as argument, assume step executor is stateless
            cur_step_output = self._run_step(task.task_id, mode=mode)

            if cur_step_output.is_last:
                result_output = cur_step_output
                break

        return self.finalize_response(task.task_id, result_output)

    async def _achat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
        mode: ChatResponseMode = ChatResponseMode.WAIT,
    ) -> AGENT_CHAT_RESPONSE_TYPE:
        """Chat with step executor."""
        if chat_history is not None:
            self.memory.set(chat_history)
        task = self.create_task(message)

        result_output = None
        while True:
            # pass step queue in as argument, assume step executor is stateless
            cur_step_output = await self._arun_step(task.task_id, mode=mode)

            if cur_step_output.is_last:
                result_output = cur_step_output
                break

        return self.finalize_response(task.task_id, result_output)

    @trace_method("chat")
    def chat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
    ) -> AgentChatResponse:
        with self.callback_manager.event(
            CBEventType.AGENT_STEP,
            payload={EventPayload.MESSAGES: [message]},
        ) as e:
            chat_response = self._chat(
                message, chat_history, tool_choice, mode=ChatResponseMode.WAIT
            )
            assert isinstance(chat_response, AgentChatResponse)
            e.on_end(payload={EventPayload.RESPONSE: chat_response})
        return chat_response

    @trace_method("chat")
    async def achat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
    ) -> AgentChatResponse:
        with self.callback_manager.event(
            CBEventType.AGENT_STEP,
            payload={EventPayload.MESSAGES: [message]},
        ) as e:
            chat_response = await self._achat(
                message, chat_history, tool_choice, mode=ChatResponseMode.WAIT
            )
            assert isinstance(chat_response, AgentChatResponse)
            e.on_end(payload={EventPayload.RESPONSE: chat_response})
        return chat_response

    @trace_method("chat")
    def stream_chat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
    ) -> StreamingAgentChatResponse:
        with self.callback_manager.event(
            CBEventType.AGENT_STEP,
            payload={EventPayload.MESSAGES: [message]},
        ) as e:
            chat_response = self._chat(
                message, chat_history, tool_choice, mode=ChatResponseMode.STREAM
            )
            assert isinstance(chat_response, StreamingAgentChatResponse)
            e.on_end(payload={EventPayload.RESPONSE: chat_response})
        return chat_response

    @trace_method("chat")
    async def astream_chat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
    ) -> StreamingAgentChatResponse:
        with self.callback_manager.event(
            CBEventType.AGENT_STEP,
            payload={EventPayload.MESSAGES: [message]},
        ) as e:
            chat_response = await self._achat(
                message, chat_history, tool_choice, mode=ChatResponseMode.STREAM
            )
            assert isinstance(chat_response, StreamingAgentChatResponse)
            e.on_end(payload={EventPayload.RESPONSE: chat_response})
        return chat_response

    def undo_step(self, task_id: str) -> None:
        """Undo previous step."""
        raise NotImplementedError("undo_step not implemented")

"""ReAct agent.

Simple wrapper around AgentRunner + ReActAgentWorker.

For the legacy implementation see:
```python
from llama_index.agent.legacy.react.base import ReActAgent
```

"""
from typing import (
    Any,
    List,
    Optional,
    Sequence,
    Type,
)

from llama_index.agent.react.formatter import ReActChatFormatter
from llama_index.agent.react.output_parser import ReActOutputParser
#from llama_index.agent.react.step import ReActAgentWorker
#from llama_index.agent.runner.base import AgentRunner
from llama_index.callbacks import (
    CallbackManager,
)
from llama_index.llms.llm import LLM
from llama_index.llms.openai import OpenAI
from llama_index.llms.types import ChatMessage
from llama_index.memory.chat_memory_buffer import ChatMemoryBuffer
from llama_index.memory.types import BaseMemory
from llama_index.objects.base import ObjectRetriever
from llama_index.tools import BaseTool

DEFAULT_MODEL_NAME = "gpt-3.5-turbo-0613"


class ReActAgent(AgentRunner):
    """ReAct agent.

    Subclasses AgentRunner with a ReActAgentWorker.

    For the legacy implementation see:
    ```python
    from llama_index.agent.legacy.react.base import ReActAgent
    ```

    """

    def __init__(
        self,
        tools: Sequence[BaseTool],
        llm: LLM,
        memory: BaseMemory,
        max_iterations: int = 10,
        react_chat_formatter: Optional[ReActChatFormatter] = None,
        output_parser: Optional[ReActOutputParser] = None,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = False,
        tool_retriever: Optional[ObjectRetriever[BaseTool]] = None,
    ) -> None:
        """Init params."""
        callback_manager = callback_manager or llm.callback_manager
        step_engine = ReActAgentWorker.from_tools(
            tools=tools,
            tool_retriever=tool_retriever,
            llm=llm,
            max_iterations=max_iterations,
            react_chat_formatter=react_chat_formatter,
            output_parser=output_parser,
            callback_manager=callback_manager,
            verbose=verbose,
        )
        super().__init__(
            step_engine,
            memory=memory,
            llm=llm,
            callback_manager=callback_manager,
        )

    @classmethod
    def from_tools(
        cls,
        tools: Optional[List[BaseTool]] = None,
        tool_retriever: Optional[ObjectRetriever[BaseTool]] = None,
        llm: Optional[LLM] = None,
        chat_history: Optional[List[ChatMessage]] = None,
        memory: Optional[BaseMemory] = None,
        memory_cls: Type[BaseMemory] = ChatMemoryBuffer,
        max_iterations: int = 10,
        react_chat_formatter: Optional[ReActChatFormatter] = None,
        output_parser: Optional[ReActOutputParser] = None,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = False,
        **kwargs: Any,
    ) -> "ReActAgent":
        """Convenience constructor method from set of of BaseTools (Optional).

        NOTE: kwargs should have been exhausted by this point. In other words
        the various upstream components such as BaseSynthesizer (response synthesizer)
        or BaseRetriever should have picked up off their respective kwargs in their
        constructions.

        Returns:
            ReActAgent
        """
        llm = llm or OpenAI(model=DEFAULT_MODEL_NAME)
        if callback_manager is not None:
            llm.callback_manager = callback_manager
        memory = memory or memory_cls.from_defaults(
            chat_history=chat_history or [], llm=llm
        )
        return cls(
            tools=tools or [],
            tool_retriever=tool_retriever,
            llm=llm,
            memory=memory,
            max_iterations=max_iterations,
            react_chat_formatter=react_chat_formatter,
            output_parser=output_parser,
            callback_manager=callback_manager,
            verbose=verbose,
        )
    
    

from llama_index.response.notebook_utils import display_source_node
from IPython.display import Markdown, display

# define prompt viewing function
def display_prompt_dict(prompts_dict):
    for k, p in prompts_dict.items():
        text_md = f"**Prompt Key**: {k}<br>" f"**Text:** <br>"
        display(Markdown(text_md))
        print(p.get_template())
        display(Markdown("<br><br>"))

import os
from llama_index import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index import load_index_from_storage
from llama_index.node_parser import HierarchicalNodeParser
from llama_index.indices.postprocessor import SentenceTransformerRerank
from llama_index.retrievers import AutoMergingRetriever
from llama_index.query_engine import RetrieverQueryEngine
from llama_index.response.notebook_utils import display_response
from llama_index.tools import QueryEngineTool, ToolMetadata
#from llama_index.agent import ReActAgent
from llama_index.tools.utils import create_schema_from_function

# file_2_index = {}
# file_2_codings = {}
# file_2_engine = {}
similarity_top_k = 12

rerank = SentenceTransformerRerank(top_n=6, model="BAAI/bge-reranker-large")

class CRCRetrievalEngineToolSpec(BaseToolSpec):
    """Tool spec."""

    spec_functions = ["get_histology_fn"]

    def __init__(
        self,
        #index: BaseIndex,  # TODO typing
        doc,
        idx,
        nodes,
        auto_merging_context
    ) -> None:
        """Initialize with parameters."""
        file_path = Path(doc.metadata["file_path"])
        file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)        
        vi_out_path = f"./data/llamaindex_docs/{file_id}_{idx}"
        if not os.path.exists(vi_out_path):
            leaf_nodes = get_leaf_nodes(nodes)        
            storage_context = StorageContext.from_defaults() # 將 index 儲存在記憶體中
            storage_context.docstore.add_documents(nodes)
            # https://docs.llamaindex.ai/en/stable/api_reference/indices/vector_store.html#llama_index.indices.vector_store.base.VectorStoreIndex
            # 這邊我們提供 leaf_nodes 作為 embedding index 的 node，storage context 中則是所有的 nodes 包括整個結構，最後 service_context 是整個 service 用到的相關資源
            self._index = VectorStoreIndex(
                    leaf_nodes,
                    service_context=auto_merging_context,
                    storage_context=storage_context
                )
            self._index.storage_context.persist(persist_dir=vi_out_path)
        else:
            self._index = load_index_from_storage(
                StorageContext.from_defaults(persist_dir=vi_out_path),
                service_context=auto_merging_context,
            )
        self._context = auto_merging_context

    def get_histology_fn(
        self,
        query: str,
#        top_k: int,
#        filter_key_list: List[str],
#        filter_value_list: List[str],
    ) -> str:
        """Auto retrieval function.

        Useful for answering questions related to histology using the medical report

        Args:
            query (str): The query to search
        """
            # top_k (int): The number of results to retrieve
            # filter_key_list (List[str]): The list of filter keys
            # filter_value_list (List[str]): The list of filter values

        # exact_match_filters = [
        #     ExactMatchFilter(key=k, value=v)
        #     for k, v in zip(filter_key_list, filter_value_list)
        # ]
        # retriever = VectorIndexRetriever(
        #     self._index,
        #     filters=MetadataFilters(filters=exact_match_filters),
        #     top_k=top_k,
        # )
        # query_engine = RetrieverQueryEngine.from_args(retriever)

        

        retriever = self._index.as_retriever(similarity_top_k=similarity_top_k)
            # https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html
        # retriever = AutoMergingRetriever(
        #     retriever, 
        #     self._index.storage_context, 
        #     verbose=True
        # )

        #query_engine = RetrieverQueryEngine.from_args(automerging_retriever, node_postprocessors=[rerank], 
         #                                                    service_context = self._context)
        query_engine = RetrieverQueryEngine.from_args(retriever, node_postprocessors=[rerank], 
                                                             service_context = self._context)

        update_query_engine_prompts(ccr_histology_template, query_engine)
        prompts_dict = query_engine.get_prompts()
        display_prompt_dict(prompts_dict)

        response = query_engine.query(query)        
        return str(response)


def build_agent_per_doc(doc, idx, nodes, auto_merging_context, agent_document_idx_templates):
    crc_tool_spec = CRCRetrievalEngineToolSpec(doc, idx, nodes, auto_merging_context)
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)

    #print(crc_tool_spec.get_histology_fn.__doc__)    
    crc_tool= crc_tool_spec.to_tool_list(
        func_to_metadata_mapping={'get_histology_fn': ToolMetadata(
            name="get_histology",
            # description=f"""\
            #     Use this tool to look up biographical information about celebrities.
            #     The vector database schema is given below:
                
            #     {vector_store_info.json()}
                
            #     {tool_spec.auto_retrieve_fn.__doc__}
            # """,
            description=(f"""Useful for answering questions related to histology using the medical report.
The schema is given below:
{crc_tool_spec.get_histology_fn.__doc__}
""",
                         
                         ),# {doc.metadata['date']}"),
            fn_schema=create_schema_from_function("get_histology", crc_tool_spec.get_histology_fn)
        )}
    )

    agent = ReActAgent.from_tools(crc_tool, llm=agent_llm, verbose=True,
                                  system_prompt=f"""You are a specialized agent designed to answer queries about the \
cancer registry coding results based on a patient's medical reports.
You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on any prior knowledge.\
""")
    return agent, f"{file_id}_{idx}"#, summary
    
    
from llama_index.node_parser import HierarchicalNodeParser
import os

# https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents-v1.html
def build_agents(docs):
    # See https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html#
    node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 1024, 512])

    auto_merging_context = ServiceContext.from_defaults(llm=doc_idx_llm,
                                                        embed_model="local:BAAI/bge-large-en-v1.5",
                                                        node_parser=node_parser)

    # Build agents dictionary
    agents_dict = {}
    extra_info_dict = {}

    # # this is for the baseline
    # all_nodes = []

    for idx, doc in enumerate((docs)):#tqdm(docs)):
        nodes = node_parser.get_nodes_from_documents([doc])
        # all_nodes.extend(nodes)

        # ID will be base + parent
        agent, file_id = build_agent_per_doc(doc, idx, nodes, auto_merging_context, agent_document_idx_templates)#, summary = build_agent_per_doc(doc, nodes, file_id)

        agents_dict[file_id] = agent
        #extra_info_dict[file_id] = {"summary": summary, "nodes": nodes}

    return agents_dict#, extra_info_dict

agents = build_agents(docs)

response = agents['crc_fb3936e6e72593d5c7539dccfb0f06c7_1'].chat("What is the histology coding of the patient if he has colorectal cancer?")
print(str(response))                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                
    

In [ ]:
from llama_index.node_parser import HierarchicalNodeParser

node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128]
)

nodes = node_parser.get_nodes_from_documents([docs[0]])
print(len(nodes)) # 以這個例子來說，最上層是一個 node，下面有兩個 node，在往下第一個子節點有 5 個子節點，第二個子節點有三個子節點

from llama_index.node_parser import get_leaf_nodes

# 總共 8 個子節點
leaf_nodes = get_leaf_nodes(nodes)
print(len(leaf_nodes))

for leaf in leaf_nodes:
    print(len(leaf.text.split('\n')))
    for text in leaf.text.split('\n'):
        print(text)
    print(leaf.parent_node.node_id)
    print("-"*50)
print("="*50)    
leafs_by_id = {node.node_id: node for node in leaf_nodes}
for ids in leafs_by_id:    
    parent_node = leafs_by_id[ids].parent_node
    print(f'{ids} --> {parent_node.node_id}')
print(parent_node.parent_node.node_id) # 到頂了

In [ ]:
def build_agent_per_doc(doc, idx, nodes, auto_merging_context, agent_document_idx_templates):
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)        
    vi_out_path = f"./data/llamaindex_docs/{file_id}_{idx}"

    class ToolSchema(BaseModel):
        pass
    #summary_out_path = f"./data/llamaindex_docs/{file_id}_summary.pkl"
    if not os.path.exists(vi_out_path):
        leaf_nodes = get_leaf_nodes(nodes)        
        storage_context = StorageContext.from_defaults() # 將 index 儲存在記憶體中
        storage_context.docstore.add_documents(nodes)
        # https://docs.llamaindex.ai/en/stable/api_reference/indices/vector_store.html#llama_index.indices.vector_store.base.VectorStoreIndex
        # 這邊我們提供 leaf_nodes 作為 embedding index 的 node，storage context 中則是所有的 nodes 包括整個結構，最後 service_context 是整個 service 用到的相關資源
        automerging_index = VectorStoreIndex(
                leaf_nodes,
                service_context=auto_merging_context,
                storage_context=storage_context
            )
        automerging_index.storage_context.persist(persist_dir=vi_out_path)
    else:
        automerging_index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=vi_out_path),
            service_context=auto_merging_context,
        )
    
    base_retriever = automerging_index.as_retriever(similarity_top_k=similarity_top_k)
    # https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html
    automerging_retriever = AutoMergingRetriever(
        base_retriever, 
        automerging_index.storage_context, 
        verbose=True
    )

    #nodes = automerging_retriever.retrieve("What is the histology of the patient?")
    # for node in nodes:
    #     display_source_node(node, source_length=10000)
    # print("*"*50)
    # base_nodes = base_retriever.retrieve("What is the histology of the patient?")
    # for node in base_nodes:
    #     display_source_node(node, source_length=10000)
    # auto_merging_engines[doc.metadata['date']] = auto_merging_engine
    
    # build summary index
    # summary_index = SummaryIndex(nodes, service_context=service_context)
    
    query_engine_tools = []
    for template in agent_document_idx_templates.keys():
        # define query engines
        # 目前用 base 好像就可以了，用 auto merging 反而做不好??
        auto_merging_engine = RetrieverQueryEngine.from_args(automerging_retriever, node_postprocessors=[rerank], 
                                                             service_context = auto_merging_context)
        # auto_merging_engine = RetrieverQueryEngine.from_args(base_retriever, node_postprocessors=[rerank], 
        #                                                     service_context = auto_merging_context)

        update_query_engine_prompts(agent_document_idx_templates[template], auto_merging_engine)
        prompts_dict = auto_merging_engine.get_prompts()
        display_prompt_dict(prompts_dict)

        # define tools
        query_engine_tools.append(QueryEngineTool(
            query_engine=auto_merging_engine,
            metadata=ToolMetadata(
                name=f"get_{template}",
                description=(f"Useful for answering questions related to {template.replace('_', ' ')} using the medical report dated {doc.metadata['date']}"),
                fn_schema= ToolSchema
            )))

    # summary_query_engine = summary_index.as_query_engine(
    #     response_mode="tree_summarize"
    # )

    # extract a summary
    # if not os.path.exists(summary_out_path):
    #     Path(summary_out_path).parent.mkdir(parents=True, exist_ok=True)
    #     summary = str(
    #         await summary_query_engine.aquery(
    #             "Extract a concise 1-2 line summary of this document"
    #         )
    #     )
    #     pickle.dump(summary, open(summary_out_path, "wb"))
    # else:
    #     summary = pickle.load(open(summary_out_path, "rb"))

    
#         QueryEngineTool(
#             query_engine=summary_query_engine,
#             metadata=ToolMetadata(
#                 name=f"summary_tool_{file_base}",
#                 description=f"Useful for summarization questions",
#             ),
#         ),
#     ]

#     # build agent
#     function_llm = OpenAI(model="gpt-4")
#     agent = OpenAIAgent.from_tools(
#         query_engine_tools,
#         llm=function_llm,
#         verbose=True,
#         system_prompt=f"""\
# You are a specialized agent designed to answer queries about the `{file_base}.html` part of the LlamaIndex docs.
# You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on prior knowledge.\
# """,
#     )
    agent = ReActAgent.from_tools(query_engine_tools, llm=agent_llm, verbose=True,
                                  system_prompt=f"""You are a specialized agent designed to answer queries about the \
cancer registry coding results based on a patient's medical reports.
You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on any prior knowledge.\
""")
    return agent, f"{file_id}_{idx}"#, summary

In [ ]:
from llama_index.node_parser import HierarchicalNodeParser
import os

# https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents-v1.html
def build_agents(docs):
    # See https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html#
    node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128])

    auto_merging_context = ServiceContext.from_defaults(llm=doc_idx_llm,
                                                        embed_model="local:BAAI/bge-large-en-v1.5",
                                                        node_parser=node_parser)

    # Build agents dictionary
    agents_dict = {}
    extra_info_dict = {}

    # # this is for the baseline
    # all_nodes = []

    for idx, doc in enumerate((docs)):#tqdm(docs)):
        nodes = node_parser.get_nodes_from_documents([doc])
        # all_nodes.extend(nodes)

        # ID will be base + parent
        agent, file_id = build_agent_per_doc(doc, idx, nodes, auto_merging_context, agent_document_idx_templates)#, summary = build_agent_per_doc(doc, nodes, file_id)

        agents_dict[file_id] = agent
        #extra_info_dict[file_id] = {"summary": summary, "nodes": nodes}

    return agents_dict#, extra_info_dict

agents = build_agents(docs)

In [ ]:
response = agents['crc_fb3936e6e72593d5c7539dccfb0f06c7_1'].chat("What is the histology coding of the patient if he has colorectal cancer?")
print(str(response))

In [ ]:
from typing import List

from llama_index.indices.base import BaseIndex
from llama_index.query_engine import RetrieverQueryEngine
from llama_index.retrievers import VectorIndexRetriever
from llama_index.tools.tool_spec.base import BaseToolSpec
from llama_index.vector_stores.types import ExactMatchFilter, MetadataFilters
from llama_index.node_parser import get_leaf_nodes
from llama_index.indices.postprocessor import SentenceTransformerRerank
from llama_index import StorageContext, load_index_from_storage
import pickle
from llama_index.tools import QueryEngineTool, ToolMetadata
import json
from llama_index.tools.utils import create_schema_from_function
from llama_index.agent import ReActAgent

rerank = SentenceTransformerRerank(top_n=6, model="BAAI/bge-reranker-large")

class CRCRetrievalEngineToolSpec(BaseToolSpec):
    """Tool spec."""

    spec_functions = ["get_histology_fn"]

    def __init__(
        self,
        #index: BaseIndex,  # TODO typing
        doc,
        idx,
        nodes,
        auto_merging_context
    ) -> None:
        """Initialize with parameters."""
        file_path = Path(doc.metadata["file_path"])
        file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)        
        vi_out_path = f"./data/llamaindex_docs/{file_id}_{idx}"
        if not os.path.exists(vi_out_path):
            leaf_nodes = get_leaf_nodes(nodes)        
            storage_context = StorageContext.from_defaults() # 將 index 儲存在記憶體中
            storage_context.docstore.add_documents(nodes)
            # https://docs.llamaindex.ai/en/stable/api_reference/indices/vector_store.html#llama_index.indices.vector_store.base.VectorStoreIndex
            # 這邊我們提供 leaf_nodes 作為 embedding index 的 node，storage context 中則是所有的 nodes 包括整個結構，最後 service_context 是整個 service 用到的相關資源
            self._index = VectorStoreIndex(
                    leaf_nodes,
                    service_context=auto_merging_context,
                    storage_context=storage_context
                )
            self._index.storage_context.persist(persist_dir=vi_out_path)
        else:
            self._index = load_index_from_storage(
                StorageContext.from_defaults(persist_dir=vi_out_path),
                service_context=auto_merging_context,
            )
        self._context = auto_merging_context

    def get_histology_fn(
        self,
        query: str,
#        top_k: int,
#        filter_key_list: List[str],
#        filter_value_list: List[str],
    ) -> str:
        """Extract the histology information.

        Useful for answering questions related to the histology of colorectal cancer patients
        
        Args:
            query (str): The query for the extraction purpose
        """
            # top_k (int): The number of results to retrieve
            # filter_key_list (List[str]): The list of filter keys
            # filter_value_list (List[str]): The list of filter values

        # exact_match_filters = [
        #     ExactMatchFilter(key=k, value=v)
        #     for k, v in zip(filter_key_list, filter_value_list)
        # ]
        # retriever = VectorIndexRetriever(
        #     self._index,
        #     filters=MetadataFilters(filters=exact_match_filters),
        #     top_k=top_k,
        # )
        # query_engine = RetrieverQueryEngine.from_args(retriever)

        

        retriever = self._index.as_retriever(similarity_top_k=similarity_top_k)
            # https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html
        # retriever = AutoMergingRetriever(
        #     retriever, 
        #     self._index.storage_context, 
        #     verbose=True
        # )

        #query_engine = RetrieverQueryEngine.from_args(automerging_retriever, node_postprocessors=[rerank], 
         #                                                    service_context = self._context)
        query_engine = RetrieverQueryEngine.from_args(retriever, node_postprocessors=[rerank], 
                                                             service_context = self._context)

        update_query_engine_prompts(ccr_histology_template, query_engine)
        prompts_dict = query_engine.get_prompts()
        display_prompt_dict(prompts_dict)

        response = query_engine.query(query)        
        return str(response)

def build_agent_per_doc(doc, idx, nodes, auto_merging_context, agent_document_idx_templates):
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)
    crc_tool_spec = CRCRetrievalEngineToolSpec(doc, idx, nodes, auto_merging_context)

    histology_analysis_file_path = f".agent/histology/{file_id}_{idx}"
    # determine the histology
    if not os.path.exists(histology_analysis_file_path):
        Path(histology_analysis_file_path).parent.mkdir(parents=True, exist_ok=True)
        results = crc_tool_spec.get_histology_fn("""Your task is to provide the histology coding for patients diagnosed with colorectal cancer. Your response should be based on the nature of the report:
- In cases where the report is not a pathology report or lacks sufficient information to ascertain the histology type, your response should be 'NA'. This indicates that histology coding cannot be determined from the provided document.
- If the report is a pathology report that confirms a diagnosis of colorectal cancer, you are required to analyze the details within and respond with the most appropriate histology code.""")
        results = json.loads(results)        
        pickle.dump(results, open(histology_analysis_file_path, "wb"))
    else:
        results = pickle.load(open(histology_analysis_file_path, "rb"))

    if results['histology'] != "NaN":
        
        crc_tool= crc_tool_spec.to_tool_list(
            func_to_metadata_mapping={'get_histology_fn': ToolMetadata(
                name="get_histology",
                # description=f"""\
                #     Use this tool to look up biographical information about celebrities.
                #     The vector database schema is given below:
                    
                #     {vector_store_info.json()}
                    
                #     {tool_spec.auto_retrieve_fn.__doc__}
                # """,
                description=(f"""Useful for answering questions related to the histology of colorectal cancer patients.
The schema is given below:
{crc_tool_spec.get_histology_fn.__doc__}
""",
                            
                            ),# {doc.metadata['date']}"),
                fn_schema=create_schema_from_function("get_histology", crc_tool_spec.get_histology_fn)
            )}
        )

        agent = ReActAgent.from_tools(crc_tool, llm=agent_llm, verbose=True,
                                    system_prompt=f"""You are a specialized agent designed to answer queries about the \
    cancer registry coding results based on a patient's medical reports.
    You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on any prior knowledge.\
    """)
        return agent, f"{file_id}_{idx}", results#, summary
    else:
        return None, None, None
    
    
from llama_index.node_parser import HierarchicalNodeParser
import os

# https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents-v1.html
def build_agents(docs):
    # See https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html#
    node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 1024, 512])

    auto_merging_context = ServiceContext.from_defaults(llm=doc_idx_llm,
                                                        embed_model="local:BAAI/bge-large-en-v1.5",
                                                        node_parser=node_parser)

    # Build agents dictionary
    agents_dict = {}
    extra_info_dict = {}

    # # this is for the baseline
    # all_nodes = []

    for idx, doc in enumerate((docs)):#tqdm(docs)):
        nodes = node_parser.get_nodes_from_documents([doc])
        # all_nodes.extend(nodes)

        # ID will be base + parent
        agent, file_id, results = build_agent_per_doc(doc, idx, nodes, auto_merging_context, agent_document_idx_templates)#, summary = build_agent_per_doc(doc, nodes, file_id)

        if agent is not None:
            agents_dict[file_id] = agent
            extra_info_dict[file_id] = {"summary": results, "nodes": nodes}

    return agents_dict, extra_info_dict

agents_dict, extra_info_dict = build_agents(docs)

In [ ]:
response = agents['crc_fb3936e6e72593d5c7539dccfb0f06c7_1'].chat("What is the histology coding of the patient if he has colorectal cancer?")
print(str(response))                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [ ]:
agents_dict

In [ ]:
# define tool for each document agent
all_tools = []
for file_base, agent in agents_dict.items():
    summary = extra_info_dict[file_base]["summary"]
    doc_tool = QueryEngineTool(
        query_engine=agent,
        metadata=ToolMetadata(
            name=f"tool_{file_base}",
            description=f'Provide a search tool for the document that can be summarized as follows: {summary["explan"]}',
        ),
    )
    all_tools.append(doc_tool)
all_tools[0].metadata

In [ ]:
from llama_index import (
    ServiceContext,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.query_engine import SubQuestionQueryEngine
from llama_index.retrievers import BaseRetriever
from llama_index.objects import (
    ObjectIndex,
    SimpleToolNodeMapping,
    ObjectRetriever,
)
from llama_index.postprocessor import CohereRerank
#agent_system_prompt = '''As a cancer registrar agent, your primary responsibility is to accurately curate the cancer registry data in compliance with user-defined requirements and the document you manage. It's imperative that your data compilation strictly adheres to the options and categories specified by users. In situations where the correct code cannot be determined due to insufficient information, or the content is unrelated to the specific cancer type in question, your response should be 'NA'. All responses must be formatted exclusively as valid JSON objects. Outputs in any format other than JSON are not acceptable. Each response should include the key 'explan', containing a string that justifies your classification choice. Additionally, incorporate other keys as required by user specifications. Remember, the structure and content of your JSON response should reflect the precision and adherence to guidelines essential for this role.'''

index_llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=2048,    
    generate_kwargs={"temperature": 0.001, "do_sample": True},
    tokenizer_name=selected_model,
    model_name=selected_model,
    is_chat_model = True,
    #system_prompt=agent_system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    # Here is how to fix the alternative error show above.
    # We transform inputs into Llama2 format here
    messages_to_prompt = messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto", 
    model_kwargs={"quantization_config": quantization_config, "pad_token_id" : 2},
)

In [ ]:
index_service_context = ServiceContext.from_defaults(
    llm=index_llm,
    embed_model="local:BAAI/bge-large-en-v1.5"
)

tool_mapping = SimpleToolNodeMapping.from_objects(all_tools)
obj_index = ObjectIndex.from_objects(
    all_tools,
    tool_mapping,
    VectorStoreIndex,
    service_context = index_service_context
)
vector_node_retriever = obj_index.as_node_retriever(similarity_top_k=10)

# define a custom retriever with reranking
class CustomRetriever(BaseRetriever):
    def __init__(self, vector_retriever, postprocessor=None):
        self._vector_retriever = vector_retriever
        self._postprocessor = postprocessor or CohereRerank(top_n=5)
        super().__init__()

    def _retrieve(self, query_bundle):
        retrieved_nodes = self._vector_retriever.retrieve(query_bundle)
        filtered_nodes = self._postprocessor.postprocess_nodes(
            retrieved_nodes, query_bundle=query_bundle
        )

        return filtered_nodes


# define a custom object retriever that adds in a query planning tool
class CustomObjectRetriever(ObjectRetriever):
    def __init__(self, retriever, object_node_mapping, all_tools, service_context=None):
        self._retriever = retriever
        self._object_node_mapping = object_node_mapping
        self._service_context = service_context

    def retrieve(self, query_bundle):
        nodes = self._retriever.retrieve(query_bundle)
        tools = [self._object_node_mapping.from_node(n.node) for n in nodes]

        sub_question_engine = SubQuestionQueryEngine.from_defaults(
            query_engine_tools=tools, service_context=self._service_context
        )
        sub_question_description = f"""\
Useful for any queries that involve comparing multiple documents. ALWAYS use this tool for comparison queries - make sure to call this \
tool with the original query. Do NOT use the other tools for any queries involving multiple documents.
"""
        sub_question_tool = QueryEngineTool(
            query_engine=sub_question_engine,
            metadata=ToolMetadata(
                name="compare_tool", description=sub_question_description
            ),
        )

        return tools + [sub_question_tool]

In [ ]:
custom_node_retriever = CustomRetriever(vector_node_retriever, rerank)

# wrap it with ObjectRetriever to return objects
custom_obj_retriever = CustomObjectRetriever(
    custom_node_retriever, tool_mapping, all_tools, service_context = index_service_context
)

In [ ]:
tmps = custom_obj_retriever.retrieve("hello")
print(len(tmps))

In [ ]:
from llama_index.agent import ReActAgent

top_agent = ReActAgent.from_tools(
    tool_retriever=custom_obj_retriever,
    system_prompt=""" \
You are an agent designed to answer queries about the documentation.
Please always use the tools provided to answer a question. Do not rely on prior knowledge.\

""",
    llm=index_llm,
    verbose=True,
)

In [ ]:
response = top_agent.query(
    "What is the histology coding of the patient if he has colorectal cancer?"
)

# Sentence Index

In [ ]:
from IPython.display import Markdown, display
# define prompt viewing function
def display_prompt_dict(prompts_dict):
    for k, p in prompts_dict.items():
        text_md = f"**Prompt Key**: {k}<br>" f"**Text:** <br>"
        display(Markdown(text_md))
        print(p.get_template())
        display(Markdown("<br><br>"))
        
def build_agent_per_doc(doc, idx, nodes, auto_merging_context, agent_document_idx_templates):
    crc_tool_spec = CRCRetrievalEngineToolSpec(doc, idx, auto_merging_context)
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)

    print(crc_tool_spec.get_histology_fn.__doc__)    
    crc_tool= crc_tool_spec.to_tool_list(
        func_to_metadata_mapping={'get_histology_fn': ToolMetadata(
            name="get_histology",
            # description=f"""\
            #     Use this tool to look up biographical information about celebrities.
            #     The vector database schema is given below:
                
            #     {vector_store_info.json()}
                
            #     {tool_spec.auto_retrieve_fn.__doc__}
            # """,
            description=(f"""Useful for answering questions related to histology using the medical report.
The schema is given below:
{crc_tool_spec.get_histology_fn.__doc__}
""",
                         
                         ),# {doc.metadata['date']}"),
            fn_schema=create_schema_from_function("get_histology", crc_tool_spec.get_histology_fn)
        )}
    )

    agent = ReActAgent.from_tools(crc_tool, llm=agent_llm, verbose=True,
                                  system_prompt=f"""You are a specialized agent designed to answer queries about the \
cancer registry coding results based on a patient's medical reports.
You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on any prior knowledge.\
""")
    return agent, f"{file_id}_{idx}"#, summary

In [ ]:
from llama_index.tools import QueryEngineTool, ToolMetadata
from typing import List
from llama_index.agent import ReActAgent
from llama_index import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.indices.base import BaseIndex
from llama_index.query_engine import RetrieverQueryEngine
from llama_index.retrievers import VectorIndexRetriever
from llama_index.tools.tool_spec.base import BaseToolSpec
from llama_index.vector_stores.types import ExactMatchFilter, MetadataFilters
from llama_index.node_parser import SentenceWindowNodeParser
from llama_index.tools.utils import create_schema_from_function

similarity_top_k = 6

class CRCRetrievalEngineToolSpec(BaseToolSpec):
    """Tool spec."""

    spec_functions = ["get_histology_fn"]

    def __init__(
        self,
        #index: BaseIndex,  # TODO typing
        doc,
        idx,
        service_context
    ) -> None:
        """Initialize with parameters."""
        file_path = Path(doc.metadata["file_path"])
        file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)        
        vi_out_path = f"./data/llamaindex_docs/{file_id}_{idx}"

        if not os.path.exists(vi_out_path):
            self._index = VectorStoreIndex.from_documents(
                [doc], service_context=service_context
            )
            self._index.storage_context.persist(persist_dir=vi_out_path)
        else:
            self._index = load_index_from_storage(
                StorageContext.from_defaults(persist_dir=vi_out_path),
                service_context=service_context,
            )
        self._context = service_context

    def get_histology_fn(
        self,
        query: str,
#        top_k: int,
#        filter_key_list: List[str],
#        filter_value_list: List[str],
    ) -> str:
        """Auto retrieval function.

        Useful for answering questions related to histology using the medical report

        Args:
            query (str): The query to search
        """
            # top_k (int): The number of results to retrieve
            # filter_key_list (List[str]): The list of filter keys
            # filter_value_list (List[str]): The list of filter values

        # exact_match_filters = [
        #     ExactMatchFilter(key=k, value=v)
        #     for k, v in zip(filter_key_list, filter_value_list)
        # ]
        # retriever = VectorIndexRetriever(
        #     self._index,
        #     filters=MetadataFilters(filters=exact_match_filters),
        #     top_k=top_k,
        # )
        # query_engine = RetrieverQueryEngine.from_args(retriever)
        
        query_engine = self._index.as_query_engine(similarity_top_k=similarity_top_k)

        update_query_engine_prompts(ccr_histology_template, query_engine)
        prompts_dict = query_engine.get_prompts()
        display_prompt_dict(prompts_dict)

        response = query_engine.query(query)        
        return str(response)
    
import os

sentence_window_size = 3
# https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents-v1.html
def build_agents(docs):
    # create the sentence window node parser w/ default settings
    node_parser = SentenceWindowNodeParser.from_defaults(
            window_size=sentence_window_size, # 往前 2 句/往後 2 句 // 目前的實作看起來有點怪怪的
            window_metadata_key="window", # The metadata key to store the sentence window under.
            original_text_metadata_key="original_text", # The metadata key to store the original sentence in.
            #sentence_splitter = split_by_sentence_tokenizer()
        )


    service_context = ServiceContext.from_defaults(llm=doc_idx_llm,
                                                        embed_model="local:BAAI/bge-large-en-v1.5",
                                                        node_parser=node_parser)

    # Build agents dictionary
    agents_dict = {}
    extra_info_dict = {}

    # # this is for the baseline
    # all_nodes = []

    for idx, doc in enumerate((docs)):#tqdm(docs)):
        nodes = node_parser.get_nodes_from_documents([doc])
        # all_nodes.extend(nodes)

        # ID will be base + parent
        agent, file_id = build_agent_per_doc(doc, idx, nodes, service_context, agent_document_idx_templates)#, summary = build_agent_per_doc(doc, nodes, file_id)

        agents_dict[file_id] = agent
        #extra_info_dict[file_id] = {"summary": summary, "nodes": nodes}

    return agents_dict#, extra_info_dict

agents = build_agents(docs)
agents    

In [ ]:
response = agents['crc_fb3936e6e72593d5c7539dccfb0f06c7_1'].chat("What is the histology coding of the patient if he has colorectal cancer?")
print(str(response))                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                